# 04 — Stage 3: GPT-4o + RAG → Faithfulness Metrics (HR/GS/CTC)

Tiap artifact 3-arm → RAG (BGE-M3, ~28 chunk) → GPT-4o (OpenRouter) → L-F-V JSON → metrik **HR/GS/CTC** (§4.1) → Friedman + Wilcoxon + bootstrap CI (§4.3). **Ini menghasilkan angka eksperimen utama thesis.**

**Prasyarat:** `02_make_artifacts` sudah jalan (`outputs/artifacts/manifest.jsonl` ada) + Secret `OPENROUTER_API_KEY`.

RAG chunk di-retrieve sekali per deteksi, dipakai ulang 3 arm (held constant). Tidak butuh GPU (semua API/CPU) — boleh runtime CPU.

## Cell 1 — Mount Drive + sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1
%cd {REPO}/scripts

## Cell 2 — Install deps + OpenRouter key

In [ ]:
%pip install -q openai sentence-transformers scipy numpy
import os
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('Key loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))

## Cell 3 — Embed knowledge base (BGE-M3, ~28 chunk) — sekali

In [ ]:
!python /content/opg-live/scripts/embed_kb.py --chunks /content/opg-live/data/kb/chunks.jsonl
# test retriever
!python /content/opg-live/scripts/retriever.py /content/opg-live/data/kb

## Cell 4 — TEST MURAH dulu: 3 deteksi × 3 arm = 9 panggilan
Pastikan GPT-4o balas JSON valid & metrik kebaca SEBELUM bakar biaya full.

In [ ]:
!python /content/opg-live/scripts/run_stage3.py --drive {DRIVE_ROOT} --limit 3

In [ ]:
# intip 1 hasil mentah
import json
rows = [json.loads(l) for l in open(f'{DRIVE_ROOT}/outputs/metrics/results.jsonl')]
print('hasil:', len(rows), '| parsed_ok:', sum(r['parsed_ok'] for r in rows))
r = rows[0]
print('arm', r['arm'], '| HR', r['HR'], 'GS', r['GS'], 'CTC', r['CTC'])
print(json.dumps(json.loads(r['raw']), indent=2)[:900])

## Cell 5 — FULL run (semua deteksi × 3 arm)
Jalankan SETELAH Cell 4 terlihat benar. ~160 deteksi × 3 = 480 panggilan. Pantau biaya OpenRouter.

In [ ]:
!python /content/opg-live/scripts/run_stage3.py --drive {DRIVE_ROOT}

## Cell 6 — Analisis: HR/GS/CTC per arm + Friedman/Wilcoxon + per-kelas (H2-H5)

In [ ]:
!python /content/opg-live/scripts/analyze_results.py --drive {DRIVE_ROOT}

## ✅ Deliverable Stage 3 — ANGKA EKSPERIMEN UTAMA
- [ ] `outputs/metrics/results.jsonl` (HR/GS/CTC per deteksi × arm)
- [ ] mean + 95% bootstrap CI per arm
- [ ] Friedman + Wilcoxon (Bonferroni) per metrik
- [ ] breakdown per-kelas → uji H2 (caries: mask>bbox), H3 (impacted: bbox>mask), H4 (periapical), H5 (deep caries: hybrid)

**Catatan:** H4 (periapical) power rendah (n kecil) → laporkan effect-size, jangan over-claim signifikansi (§4.3).

**Selesai pipeline inti.** Sisa: user-study expert rating (opsional, §4.2) + demo Phase C.